<a href="https://colab.research.google.com/github/saulo-albuquerque-phys/GWgpu-jax/blob/prior_definitions/examples/Parameter_Estimation_ripplegw_PP_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# GWgpu_jax — **PP (probability–probability) test** with IMRPhenomD injections

This notebook validates the GWgpu_jax parameter-estimation pipeline end-to-end
by running a **PP test**:

1. Draw **N injections** of source parameters **from the prior**, keeping only
   those whose optimal network SNR ≥ `SNR_MIN` (rejection sampling). The
   volumetric distance prior gives a wide spread of SNRs **above** the
   threshold (the "different SNR values" requirement).
2. For each injection: inject an **IMRPhenomD** signal into coloured Gaussian
   noise, run the **two-phase nested sampler**, and save the **posterior
   samples** + a **corner plot** into its own ID-tagged folder under
   `tests/pp_tests/<RUN_LABEL>/`.
3. For every parameter, record the **credible level of the truth**
   `p = mean(posterior < truth)`.
4. Make the **PP plot**: each parameter's empirical CDF of `p` should follow
   the diagonal if the pipeline is well-calibrated. A **provisional PP plot is
   refreshed after every new injection** so you can watch it converge; a
   combined KS p-value is reported at the end.

It is built from `Parameter_Estimation_ripplegw_injection.ipynb` and is
**Colab-compatible**, **fully configurable** (see the CONFIG cell), and
**stop-and-continue safe**: every injection checkpoints to disk, so re-running
the loop cell resumes from where it stopped (mount Google Drive for
persistence across Colab sessions).


## 1. Install GWgpu_jax (Colab only)

This cell is a no-op when GWgpu_jax is already importable (e.g. running locally
from the repo). On Colab it pins JAX to the version the CUDA plugin understands
and installs the package from GitHub.

**Auth:** provide a GitHub PAT via Colab Secrets (🔑 sidebar → add `GH_TOKEN`),
or you will be prompted. The token is scrubbed from the environment afterwards.


In [ ]:
import importlib.util, os

RUNNING_ON_COLAB = importlib.util.find_spec("google.colab") is not None
ALREADY_INSTALLED = importlib.util.find_spec("gwgpu_jax") is not None

if RUNNING_ON_COLAB and not ALREADY_INSTALLED:
    import getpass
    OWNER, REPO, BRANCH = "saulo-albuquerque-phys", "GWgpu-jax", "prior_definitions"

    # Pin JAX to the 0.4.x line Colab's CUDA plugin still supports.
    !pip uninstall -y -q jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt 2>/dev/null
    !pip install -q "jax[cuda12]==0.4.31" "jaxlib==0.4.31"

    GH_TOKEN = None
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get("GH_TOKEN")
    except Exception:
        pass
    if not GH_TOKEN:
        GH_TOKEN = getpass.getpass(f"GitHub PAT (for {OWNER}/{REPO}): ")
    os.environ["GH_TOKEN"] = GH_TOKEN

    !pip install -q "gwgpu_jax[data] @ git+https://$GH_TOKEN@github.com/{OWNER}/{REPO}.git@{BRANCH}"
    !pip install -q corner

    del os.environ["GH_TOKEN"]
    del GH_TOKEN
    print("Installed. If JAX was re-pinned, use Runtime → Restart session, then re-run from here.")
else:
    print("GWgpu_jax already importable — skipping install.")

In [ ]:
import time, json, glob
from pathlib import Path

import jax, jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# Double precision is essential for likelihood accuracy.
jax.config.update("jax_enable_x64", True)

import gwgpu_jax
print("gwgpu_jax version :", gwgpu_jax.__version__)
print("JAX devices       :", jax.devices())

## 2. Configuration — all PE features live here

Everything you would normally tune is collected in this one cell:

* **`CONFIG`** — number of injections, the **SNR threshold** (`SNR_MIN`), output
  location, data segment, sampler budget, and seeds.
* **`PARAM_BOUNDS`** — the sampled parameters and their support (also the
  injection prior support).
* **`PRIORS`** — non-uniform prior shapes (`"sin"`, `"cos"`, `"volumetric"`);
  these are used **both** to draw the injections and inside the sampler, which
  is exactly what a PP test requires.

`SNR_MIN` injects only "detectable" sources: injections are drawn from the prior
and rejected until `NUM_INJECTIONS` with optimal network SNR ≥ `SNR_MIN` are
found, so no PE is ever run on a sub-threshold signal.

For a quick smoke test set `NUM_INJECTIONS = 3`, `NUM_LIVE = 300`,
`PHASE2_INNER_STEPS = 40`. The defaults below are tuned for a real validation
run and are correspondingly slow (≈ minutes per injection on a GPU).


In [ ]:
CONFIG = dict(
    # ── PP-test size + SNR selection ────────────────────────────────────
    NUM_INJECTIONS    = 100,       # N injections (all with SNR >= SNR_MIN)
    SNR_MIN           = 10.0,      # skip / reject injections below this optimal network SNR
    MAX_DRAW_ATTEMPTS = 20000,     # safety cap on rejection-sampling draws

    # ── Output location (stop-and-continue safe) ────────────────────────
    OUTPUT_ROOT = "tests/pp_tests", # results root (relative to repo, or absolute)
    RUN_LABEL   = "pptest_imrphenomd",  # sub-folder grouping this PP-test run
    MOUNT_DRIVE = False,           # Colab: mount Google Drive for persistence
    DRIVE_ROOT  = "/content/drive/MyDrive/gwjax_pp_tests",  # used iff MOUNT_DRIVE

    # ── Waveform / data segment ─────────────────────────────────────────
    APPROXIMANT   = "IMRPhenomD",
    F_REF         = 20.0,
    DETECTORS     = ["H1", "L1"],
    DURATION      = 4.0,           # s
    SAMPLING_RATE = 2048.0,        # Hz
    F_MIN         = 20.0,          # Hz
    F_MAX         = 512.0,         # Hz

    # ── Two-phase nested sampler budget ─────────────────────────────────
    NUM_LIVE           = 1000,
    PHASE1_INNER_STEPS = 40,       # bulk phase (cheap, vmap-amortised)
    PHASE2_INNER_STEPS = 120,      # accurate tail (main cost/quality lever)
    PHASE1_DELTA_LOGZ  = -1.0,
    PHASE1_MAX_ITERS   = 1000,
    PHASE2_MAX_ITERS   = 5000,
    LOG_DLOGZ_TARGET   = -3.0,
    NUM_POSTERIOR      = 2000,

    # ── Seeds (deterministic + resumable) ───────────────────────────────
    INJ_SEED       = 12345,        # draws the N injection parameter sets
    NOISE_SEED0    = 1000,         # noise seed for injection i = NOISE_SEED0 + i
    SAMPLER_SEED0  = 7000,         # sampler key for injection i = SAMPLER_SEED0 + i
)

# tc prior: inject_signal puts the merger at tc=0 in the segment, so the prior
# is a small window around 0 (synthetic-injection convention).
TC_CENTER, TC_HALFWIDTH = 0.0, 0.05

# Sampled parameters and their support. m1, m2 are sampled independently over
# the same range; IMRPhenomD's (m1,chi_1)<->(m2,chi_2) label symmetry is folded
# heavier-first for BOTH the truth and the posterior (see helpers), keeping the
# PP test self-consistent.
PARAM_BOUNDS = {
    "m1":          (10.0, 80.0),
    "m2":          (10.0, 80.0),
    "chi_1":       (-0.9,  0.9),
    "chi_2":       (-0.9,  0.9),
    "distance":    (50.0, 1500.0),
    "inclination": (0.0,  float(jnp.pi)),
    "ra":          (0.0,  2.0 * float(jnp.pi)),
    "dec":         (-float(jnp.pi) / 2, float(jnp.pi) / 2),
    "psi":         (0.0,  float(jnp.pi)),
    "phi_c":       (0.0,  2.0 * float(jnp.pi)),
    "tc":          (TC_CENTER - TC_HALFWIDTH, TC_CENTER + TC_HALFWIDTH),
}
FIXED_PARAMS = {}

# Non-uniform priors — used to draw injections AND inside the sampler.
PRIORS = {
    "inclination": "sin",          # p(theta) ∝ sin theta  (isotropic orientation)
    "dec":         "cos",          # p(dec)   ∝ cos dec     (isotropic sky)
    "distance":    "volumetric",   # p(d)     ∝ d^2         (uniform in volume → SNR spread)
}

PARAM_NAMES = list(PARAM_BOUNDS.keys())
print(f"sampling dimension : {len(PARAM_NAMES)}")
print(f"parameters         : {PARAM_NAMES}")
print(f"SNR threshold      : {CONFIG['SNR_MIN']}")

## 3. Output directory (and optional Google Drive)

Resolves where results are written. On Colab, set `MOUNT_DRIVE = True` in CONFIG
so the per-injection checkpoints survive a runtime restart — that is what makes
"stop and continue" work across sessions.


In [ ]:
if CONFIG["MOUNT_DRIVE"] and RUNNING_ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    output_root = Path(CONFIG["DRIVE_ROOT"])
else:
    output_root = Path(CONFIG["OUTPUT_ROOT"])

RUN_DIR = output_root / CONFIG["RUN_LABEL"]
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Results will be written to:", RUN_DIR.resolve())

# Persist the configuration alongside the results for reproducibility.
with open(RUN_DIR / "config.json", "w") as f:
    json.dump({"CONFIG": CONFIG, "PARAM_BOUNDS": PARAM_BOUNDS,
               "PRIORS": PRIORS, "FIXED_PARAMS": FIXED_PARAMS}, f, indent=2)

## 4. Waveform model and optimal-SNR helper

`optimal_network_snr` builds the projected IMRPhenomD signal and returns the
network optimal SNR — it needs only the PSD (no noise realisation). We also
build a **vmapped, jitted** batch version (`batched_network_snr`) so a whole
batch of candidate injections is scored at once while rejection-sampling.


In [ ]:
waveform_fn = gwgpu_jax.build_ripplegw_waveform_fn(
    CONFIG["APPROXIMANT"], f_ref=CONFIG["F_REF"],
)

# A single reusable network for SNR-only evaluations (project/optimal_snr do not
# mutate it, so it is safe to reuse across all injections at draw time).
_snr_grid = gwgpu_jax.TimeFrequencyGrid(
    duration=CONFIG["DURATION"], sampling_rate=CONFIG["SAMPLING_RATE"],
    f_min=CONFIG["F_MIN"], f_max=CONFIG["F_MAX"],
)
_snr_network = gwgpu_jax.Network.from_names(CONFIG["DETECTORS"], _snr_grid)
_snr_freqs = _snr_grid.frequency_domain_array


def _optimal_network_snr_jax(truth):
    """Network optimal SNR as a JAX scalar (vmap/jit-friendly; no noise)."""
    hp, hc = waveform_fn(truth, _snr_freqs)
    h_dict = _snr_network.project_waveform(
        hp, hc, truth["ra"], truth["dec"], truth["psi"], gmst=0.0,
    )
    return _snr_network.network_optimal_snr(h_dict)


def optimal_network_snr(truth):
    """Optimal network SNR of a single IMRPhenomD injection (Python float).

    Uses a fresh network each call so it is always safe to call eagerly, even
    after ``batched_network_snr`` has run on the shared network.
    """
    net = gwgpu_jax.Network.from_names(CONFIG["DETECTORS"], _snr_grid)
    hp, hc = waveform_fn(truth, _snr_freqs)
    h_dict = net.project_waveform(hp, hc, truth["ra"], truth["dec"], truth["psi"], gmst=0.0)
    return float(net.network_optimal_snr(h_dict))


# Batched scorer: maps over a dict whose leaves all have a leading batch axis.
# vmap+jit makes scoring a whole candidate batch ~hundreds of times faster than
# a Python loop (verified bit-identical to per-injection evaluation).
batched_network_snr = jax.jit(jax.vmap(_optimal_network_snr_jax))


def fold_heavier_first_batch(particles):
    """Fold a batch of prior draws to m1>=m2, carrying each spin with its mass.

    ripplegw IMRPhenomD is NOT exactly symmetric under (m1,chi_1)<->(m2,chi_2),
    so we fold BEFORE scoring/injecting — the pre-filter SNR is then exactly the
    SNR of the signal that gets injected, and truths match the heavier-first
    convention used to fold the posterior later.
    """
    m1, m2 = particles["m1"], particles["m2"]
    c1, c2 = particles["chi_1"], particles["chi_2"]
    swap = m2 > m1
    out = dict(particles)
    out["m1"], out["m2"] = jnp.where(swap, m2, m1), jnp.where(swap, m1, m2)
    out["chi_1"], out["chi_2"] = jnp.where(swap, c2, c1), jnp.where(swap, c1, c2)
    return out

## 5. Draw the N injections from the prior (SNR ≥ `SNR_MIN`)

We draw parameter sets from the prior and keep only those with optimal network
SNR ≥ `SNR_MIN`, until `NUM_INJECTIONS` are collected, **once** with a fixed
seed, then cache them (with their SNRs) to `injections.json`. On resume we
reload that file, so the exact same injections are always used. Truths are
folded **heavier-first** (`m1 ≥ m2`, carrying each spin with its mass) to match
how the posterior is folded later.


In [ ]:
INJ_FILE = RUN_DIR / "injections.json"
if INJ_FILE.exists():
    injections = json.load(open(INJ_FILE))
    print(f"Loaded {len(injections)} cached injections from {INJ_FILE}")
else:
    prior_specs = gwgpu_jax.resolve_priors(PARAM_BOUNDS, PRIORS)
    injections, attempts = [], 0
    key = jax.random.PRNGKey(CONFIG["INJ_SEED"])
    batch = max(1024, CONFIG["NUM_INJECTIONS"])
    while len(injections) < CONFIG["NUM_INJECTIONS"] and attempts < CONFIG["MAX_DRAW_ATTEMPTS"]:
        key, sub = jax.random.split(key)
        particles = fold_heavier_first_batch(gwgpu_jax.sample_prior(sub, batch, prior_specs)[0])
        snrs = np.asarray(batched_network_snr(particles))   # vmapped, fast; scored on folded draws
        attempts += batch
        for j in np.where(snrs >= CONFIG["SNR_MIN"])[0]:
            truth = {name: float(np.asarray(particles[name])[j]) for name in PARAM_NAMES}
            truth.update(FIXED_PARAMS)
            injections.append({"truth": truth, "snr": float(snrs[j])})
            if len(injections) >= CONFIG["NUM_INJECTIONS"]:
                break
    if len(injections) < CONFIG["NUM_INJECTIONS"]:
        print(f"WARNING: only {len(injections)} injections found in "
              f"{attempts} draws (raise MAX_DRAW_ATTEMPTS or lower SNR_MIN).")
    json.dump(injections, open(INJ_FILE, "w"), indent=2)
    acc = len(injections) / max(attempts, 1)
    print(f"Collected {len(injections)} injections in {attempts} draws "
          f"(acceptance {acc:.1%}); cached to {INJ_FILE}")

snrs = [inj["snr"] for inj in injections]
print(f"injection SNRs: min {min(snrs):.1f}, median {np.median(snrs):.1f}, max {max(snrs):.1f}")
print("example injection [0]:",
      {k: round(v, 3) for k, v in injections[0]["truth"].items()})

## 6. Per-injection PE + PP-plot helpers (with checkpointing)

`run_one_injection` builds a fresh network, injects the IMRPhenomD signal into
Gaussian noise, runs the two-phase sampler, folds the posterior heavier-first,
saves everything into an ID-tagged folder, and returns the per-parameter
credible levels of the truth. It **guards `SNR_MIN`**: if a signal is somehow
below threshold it is recorded as skipped and **no PE is run**.

Each injection's folder (`inj_NNN/`) holds:
* `posterior_samples.npz` — folded posterior for every parameter,
* `corner.png` — corner plot with the truth overlaid,
* `result.json` — truth, network SNR, p-values, logZ, ESS, timing, and a
  `completed: true` flag used to skip the injection on resume.

`make_pp_plot` is reused both for the live provisional plot and the final plot.


In [ ]:
def credible_levels(posterior, truth):
    """p = mean(posterior < truth) for each parameter (the PP-test statistic)."""
    return {n: float(np.mean(np.asarray(posterior[n]) < truth[n])) for n in PARAM_NAMES}


def _save_corner(posterior, truth, outfile):
    try:
        import corner
    except ImportError:
        return
    data = np.column_stack([np.asarray(posterior[n]) for n in PARAM_NAMES])
    fig = corner.corner(
        data, labels=PARAM_NAMES, truths=[truth[n] for n in PARAM_NAMES],
        range=[PARAM_BOUNDS[n] for n in PARAM_NAMES],
        quantiles=[0.16, 0.5, 0.84], show_titles=True, title_kwargs={"fontsize": 8},
    )
    fig.set_size_inches(13, 13)
    fig.savefig(outfile, dpi=90, bbox_inches="tight")
    plt.close(fig)


def run_one_injection(i, inj, *, force=False):
    """Run PE for injection i. Returns (record, status).

    status is one of: "ran", "skipped_done", "skipped_low_snr".
    """
    truth = inj["truth"]
    inj_dir = RUN_DIR / f"inj_{i:03d}"
    res_file = inj_dir / "result.json"
    if res_file.exists() and not force:
        rec = json.load(open(res_file))
        if rec.get("completed"):
            return rec, "skipped_done"
    inj_dir.mkdir(parents=True, exist_ok=True)

    # 1. Fresh network + coloured Gaussian noise + IMRPhenomD injection.
    grid = gwgpu_jax.TimeFrequencyGrid(
        duration=CONFIG["DURATION"], sampling_rate=CONFIG["SAMPLING_RATE"],
        f_min=CONFIG["F_MIN"], f_max=CONFIG["F_MAX"],
    )
    network = gwgpu_jax.Network.from_names(CONFIG["DETECTORS"], grid)
    network.generate_noise(seed=CONFIG["NOISE_SEED0"] + i)

    hp, hc = waveform_fn(truth, grid.frequency_domain_array)
    # Project with the SAME tc+dt_ifo convention as the sampler's likelihood, so
    # the injected merger time equals truth["tc"]. network.project_waveform applies
    # only the geometric delay dt_ifo (NOT tc), so it would place the signal at
    # tc=0 and tc could never be recovered.
    gmst_inj = float(network.gmst)
    freqs_inj = grid.frequency_domain_array
    h_dict = {}
    for ifo in network.interferometers:
        Fp, Fc = ifo.antenna_pattern(truth["ra"], truth["dec"], truth["psi"], gmst_inj)
        dt_ifo = ifo.time_delay_from_geocenter(truth["ra"], truth["dec"], gmst_inj)
        h_dict[ifo.name] = gwgpu_jax.waveform_projection_fd(
            hp, hc, Fp, Fc, freqs_inj, truth["tc"] + dt_ifo)
    net_snr = float(network.network_optimal_snr(h_dict))

    # SNR-threshold guard: never run PE on a sub-threshold signal.
    if net_snr < CONFIG["SNR_MIN"]:
        rec = dict(injection_id=f"inj_{i:03d}", index=i, completed=False,
                   skipped_low_snr=True, truth=truth, network_snr=net_snr)
        json.dump(rec, open(res_file, "w"), indent=2)
        return rec, "skipped_low_snr"

    network.inject_signal(h_dict, domain="fd")

    # 2. Two-phase nested sampler with the SAME priors used to draw the truth.
    sampler = gwgpu_jax.GWgpu_jaxTwoPhaseNestedSampler(
        network=network, waveform_fn=waveform_fn,
        param_bounds=PARAM_BOUNDS, fixed_params=FIXED_PARAMS,
        priors=PRIORS, gmst=None,
    )
    t0 = time.perf_counter()
    result = sampler.run_two_phase(
        rng_key                     = jax.random.PRNGKey(CONFIG["SAMPLER_SEED0"] + i),
        num_live                    = CONFIG["NUM_LIVE"],
        phase1_num_inner_steps      = CONFIG["PHASE1_INNER_STEPS"],
        phase2_num_inner_steps      = CONFIG["PHASE2_INNER_STEPS"],
        phase1_num_delete           = max(1, CONFIG["NUM_LIVE"] // 20),
        phase1_delta_logz_threshold = CONFIG["PHASE1_DELTA_LOGZ"],
        phase1_max_iterations       = CONFIG["PHASE1_MAX_ITERS"],
        phase2_num_delete           = 1,
        phase2_max_iterations       = CONFIG["PHASE2_MAX_ITERS"],
        log_dlogz_target            = CONFIG["LOG_DLOGZ_TARGET"],
        num_posterior_samples       = CONFIG["NUM_POSTERIOR"],
        verbose                     = False,
    )
    elapsed = time.perf_counter() - t0

    # 3. Fold posterior heavier-first (matches the folded truth).
    post = {n: np.asarray(result.posterior_samples[n]) for n in PARAM_NAMES}
    swap = post["m2"] > post["m1"]
    post["m1"], post["m2"] = np.where(swap, post["m2"], post["m1"]), np.where(swap, post["m1"], post["m2"])
    post["chi_1"], post["chi_2"] = np.where(swap, post["chi_2"], post["chi_1"]), np.where(swap, post["chi_1"], post["chi_2"])

    # 4. Save posterior, corner plot, and the result record.
    np.savez_compressed(inj_dir / "posterior_samples.npz", **post)
    _save_corner(post, truth, inj_dir / "corner.png")

    p_levels = credible_levels(post, truth)
    rec = dict(
        injection_id=f"inj_{i:03d}", index=i, completed=True, skipped_low_snr=False,
        truth=truth, network_snr=net_snr, p_values=p_levels,
        logZ=float(result.logZ), logZ_err=float(result.logZ_err),
        ess=float(result.ess), n_iterations=int(result.n_iterations),
        elapsed_s=elapsed,
    )
    json.dump(rec, open(res_file, "w"), indent=2)
    return rec, "ran"


def gather_completed():
    """Load every completed result.json from the run directory."""
    recs = [json.load(open(rf)) for rf in
            sorted(glob.glob(str(RUN_DIR / "inj_*" / "result.json")))]
    return [r for r in recs if r.get("completed")]


def make_pp_plot(recs, save_path=None, ax=None, title_suffix=""):
    """PP plot from a list of completed records. Returns (fig, ks_p, combined_p)."""
    from scipy import stats
    N = len(recs)
    pp = {n: np.sort([r["p_values"][n] for r in recs]) for n in PARAM_NAMES}
    x = np.linspace(0, 1, 200)

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(8, 8))
    else:
        fig = ax.figure
    ax.clear()
    # 1/2/3-sigma confidence bands for the empirical CDF under the null (binomial).
    for sig, alpha in [(1, 0.3), (2, 0.2), (3, 0.1)]:
        edge = stats.norm.cdf(sig)
        lo = stats.binom.ppf(1 - edge, N, x) / N
        hi = stats.binom.ppf(edge, N, x) / N
        ax.fill_between(x, lo, hi, color="0.5", alpha=alpha, lw=0)
    ax.plot([0, 1], [0, 1], "k--", lw=1)

    ks_p = {}
    for n in PARAM_NAMES:
        emp = np.searchsorted(pp[n], x, side="right") / N
        ks_p[n] = stats.kstest(pp[n], "uniform").pvalue
        ax.plot(x, emp, lw=1.5, label=f"{n} (p={ks_p[n]:.2f})")
    combined = stats.combine_pvalues(list(ks_p.values()), method="fisher").pvalue

    ax.set_xlabel("credible level  p"); ax.set_ylabel("fraction of injections < p")
    ax.set_title(f"PP test — N={N}, combined p = {combined:.3f}{title_suffix}")
    ax.legend(fontsize=8, loc="upper left", ncol=2)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    if save_path is not None:
        fig.savefig(save_path, dpi=130, bbox_inches="tight")
    return fig, ks_p, combined

## 7. Run the PP-test loop — provisional plot after each injection

Run this cell to process all injections. Already-completed injections are
skipped instantly (loaded from disk), so you can **interrupt the kernel and
re-run this cell** to continue. After every newly-run injection the
**provisional PP plot** is refreshed (and saved to `pp_plot.png`) so you can
watch calibration build up. A `manifest.csv` summarising every completed
injection (SNR, ESS, logZ) is refreshed at the end.


In [ ]:
import csv
from IPython.display import clear_output

n_ran, n_skipped = 0, 0
live_fig = None
for i in range(len(injections)):
    rec, status = run_one_injection(i, injections[i])
    if status == "skipped_done":
        n_skipped += 1
        continue
    if status == "skipped_low_snr":
        print(f"[{i+1:3d}/{len(injections)}] inj_{i:03d}  SNR={rec['network_snr']:.1f} "
              f"< SNR_MIN — PE skipped.")
        continue

    n_ran += 1
    completed = gather_completed()
    if len(completed) >= 2:
        clear_output(wait=True)
        if live_fig is None:
            live_fig, _ax = plt.subplots(figsize=(8, 8))
        make_pp_plot(completed, save_path=RUN_DIR / "pp_plot.png",
                     ax=live_fig.axes[0], title_suffix="  (provisional)")
        from IPython.display import display
        display(live_fig)
    print(f"[{i+1:3d}/{len(injections)}] inj_{i:03d}  ran  "
          f"SNR={rec['network_snr']:5.1f}  ESS={rec['ess']:6.1f}  "
          f"logZ={rec['logZ']:+8.1f}  ({rec['elapsed_s']:.0f}s)  "
          f"| completed so far: {len(completed)}")

print(f"\nDone. {n_ran} newly run, {n_skipped} already complete.")

# Refresh the manifest from every completed result.json.
manifest_rows = [dict(injection_id=r["injection_id"], network_snr=r["network_snr"],
                      ess=r["ess"], logZ=r["logZ"], n_iterations=r["n_iterations"],
                      elapsed_s=r.get("elapsed_s", float("nan")))
                 for r in gather_completed()]
if manifest_rows:
    with open(RUN_DIR / "manifest.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(manifest_rows[0].keys()))
        w.writeheader(); w.writerows(manifest_rows)
    msnrs = [m["network_snr"] for m in manifest_rows]
    print(f"manifest.csv: {len(manifest_rows)} injections, "
          f"SNR range {min(msnrs):.1f}–{max(msnrs):.1f}")

## 8. Final PP plot — validate the pipeline

Re-draws the PP plot from **all** completed injections. A well-calibrated
pipeline produces curves that stay inside the grey confidence bands. The legend
shows each parameter's KS p-value against Uniform(0,1); the title shows the
**combined** p-value (Fisher's method) — values that are not tiny indicate the
ensemble is consistent with perfect calibration.


In [ ]:
recs = gather_completed()
assert len(recs) > 0, "No completed injections found — run the loop cell first."
print(f"Final PP plot from {len(recs)} completed injections.")

fig, ks_p, combined = make_pp_plot(recs, save_path=RUN_DIR / "pp_plot.png")
plt.show()
print("Saved", RUN_DIR / "pp_plot.png")
print("per-parameter KS p-values:", {k: round(v, 3) for k, v in ks_p.items()})
print(f"combined p-value (Fisher): {combined:.4f}")

## Notes

- **Stop & continue:** interrupt the loop cell whenever; re-running it skips
  completed injections (`result.json` with `completed: true`). On Colab set
  `MOUNT_DRIVE = True` so checkpoints survive a runtime restart.
- **SNR selection:** injections are rejection-sampled to optimal network
  SNR ≥ `SNR_MIN`, and a guard in `run_one_injection` never runs PE below it.
  This conditions on detectable sources; the volumetric distance prior still
  gives a broad SNR spread above the threshold (see `manifest.csv`).
- **Cost vs. quality:** `PHASE2_INNER_STEPS` and `NUM_LIVE` are the main levers.
  For a quick check drop `NUM_INJECTIONS`, `NUM_LIVE`, and `PHASE2_INNER_STEPS`.
- **Interpretation:** curves inside the bands and a non-tiny combined p-value ⇒
  the sampler + priors are well-calibrated. A consistent diagonal offset for one
  parameter points to a bias in that parameter's prior/likelihood handling.
